# Check GPU

If this fails or says no GPU, go to:

Runtime → Change runtime type → T4 GPU / GPU

In [ ]:
!nvidia-smi

# Mount Google Drive

What to do beforehand:
1. Create folder in our own drive "MM_Project (local)"
2. Find the shared folder "MM_Project" & add Shortcut to your local folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# Set important paths

Change BRANCH_NAME.

In [ ]:
from pathlib import Path

# ====== CHANGE THESE ======
GITHUB_REPO_URL = "https://github.com/selina714/MM_ImageEnhancement.git"
BRANCH_NAME = "yourbranch"   # change this

# Based on your Drive structure:
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/MM_Project (local)/MM_Project/datasets/low-light")
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/MM_Project (local)/MM_Project/checkpoints")

# Colab local paths
PROJECT_DIR = Path("/content/MM_Project")
LOCAL_DATASET_ROOT = Path("/content/datasets/low-light")
NAFNET_DATA_ROOT = Path("/content/nafnet_data")

print("GitHub repo:", GITHUB_REPO_URL)
print("Branch:", BRANCH_NAME)
print("Drive dataset path:", DRIVE_DATASET_ROOT)
print("Drive checkpoint path:", DRIVE_CHECKPOINT_ROOT)
print("Local dataset path:", LOCAL_DATASET_ROOT)
print("NAFNet dataset path:", NAFNET_DATA_ROOT)

# Verify Drive dataset path

In [ ]:
print("Dataset path exists:", DRIVE_DATASET_ROOT.exists())
print("Checkpoint path exists:", DRIVE_CHECKPOINT_ROOT.exists())

if DRIVE_DATASET_ROOT.exists():
    print("\nDataset folder contents:")
    !find "$DRIVE_DATASET_ROOT" -maxdepth 2 -type d | head -30
else:
    print("Dataset path not found. Searching for low-light folder...")
    !find /content/drive/MyDrive -type d -name "low-light" 2>/dev/null | head -20

# Clone your GitHub repo

In [ ]:
%cd /content

if PROJECT_DIR.exists():
    print("Project folder already exists. Removing old copy.")
    !rm -rf "$PROJECT_DIR"

!git clone "$GITHUB_REPO_URL" "$PROJECT_DIR"

%cd "$PROJECT_DIR"
!git fetch origin
!git switch "$BRANCH_NAME"

!git status

# Check project structure

You should see:

- NAFNet
- configs
- helper_scripts
- notebooks
- project_docs

In [ ]:
%cd "$PROJECT_DIR"

print("Project root:")
!ls

print("\nNAFNet folder:")
!ls NAFNet | head

print("\nConfigs:")
!ls configs

print("\nHelper scripts:")
!ls helper_scripts

# Install dependencies

If this cell fails, stop and check the error. Do not continue to training yet.

In [ ]:
%cd "$PROJECT_DIR/NAFNet"

!pip install -r requirements.txt
!python setup.py develop --no_cuda_ext

%cd "$PROJECT_DIR"

# Copy dataset from Drive to Colab local storage

This avoids slow training directly from Google Drive.

#### Expected:

/content/datasets/low-light/train

/content/datasets/low-light/val

/content/datasets/low-light/test

In [ ]:
%cd /content

!mkdir -p /content/datasets

if LOCAL_DATASET_ROOT.exists():
    print("Local dataset already exists. Removing old copy.")
    !rm -rf "$LOCAL_DATASET_ROOT"

print("Copying dataset from Drive to Colab local storage...")
!cp -r "$DRIVE_DATASET_ROOT" "$LOCAL_DATASET_ROOT"

print("Done copying.")
!find "$LOCAL_DATASET_ROOT" -maxdepth 2 -type d

# Check original TA dataset

#### Expected output:

[train]
Input files: 30000
GT files:    30000
Pairs:       30000
Status: OK

[val]
Input files: 1000
GT files:    1000
Pairs:       1000
Status: OK

[test]
Input files: 1000
GT files:    0
Status: OK

In [ ]:
%cd "$PROJECT_DIR"

!python helper_scripts/check_lowlight_dataset.py --root "$LOCAL_DATASET_ROOT"

# Prepare NAFNet-style dataset

This creates:

/content/nafnet_data/train/lq

/content/nafnet_data/train/gt

/content/nafnet_data/val/lq

/content/nafnet_data/val/gt

/content/nafnet_data/test/lq

# Do not run full training yet

Before full training, we need to confirm these three things:

1. NAFNet accepts .webp files through PairedImageDataset.
2. The config format matches your specific NAFNet version.
3. The debug training runs for at least a few iterations.

In [ ]:
%cd "$PROJECT_DIR"

if NAFNET_DATA_ROOT.exists():
    print("Removing old prepared NAFNet dataset.")
    !rm -rf "$NAFNET_DATA_ROOT"

!python helper_scripts/prepare_lowlight_for_nafnet.py \
  --src "$LOCAL_DATASET_ROOT" \
  --dst "$NAFNET_DATA_ROOT" \
  --symlink

print("\nPrepared dataset folders:")
!find "$NAFNET_DATA_ROOT" -maxdepth 3 -type d

print("\nSample training LQ files:")
!find "$NAFNET_DATA_ROOT/train/lq" -type f | head

print("\nSample training GT files:")
!find "$NAFNET_DATA_ROOT/train/gt" -type f | head

# Verify WebP loading

In [ ]:
from pathlib import Path
from PIL import Image

sample_lq = next((NAFNET_DATA_ROOT / "train" / "lq").glob("*.webp"))
sample_gt = next((NAFNET_DATA_ROOT / "train" / "gt").glob("*.webp"))

lq_img = Image.open(sample_lq)
gt_img = Image.open(sample_gt)

print("LQ sample:", sample_lq)
print("GT sample:", sample_gt)
print("LQ:", lq_img.mode, lq_img.size)
print("GT:", gt_img.mode, gt_img.size)

# Preview one pair

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

lq = Image.open(sample_lq)
gt = Image.open(sample_gt)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(lq)
plt.title("Low-light input")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(gt)
plt.title("Ground truth")
plt.axis("off")

plt.show()

# Run debug training only

In [ ]:
%cd "$PROJECT_DIR/NAFNet"

!python basicsr/train.py -opt ../configs/train_lowlight_debug.yml

%cd "$PROJECT_DIR"

# Check debug experiment outputs

In [ ]:
DEBUG_EXP_DIR = PROJECT_DIR / "NAFNet" / "experiments" / "lowlight_nafnet_debug"

print("Debug experiment exists:", DEBUG_EXP_DIR.exists())

if DEBUG_EXP_DIR.exists():
    !find "$DEBUG_EXP_DIR" -maxdepth 3 -type d
    print("\nFiles:")
    !find "$DEBUG_EXP_DIR" -maxdepth 4 -type f | head -50
else:
    print("Debug experiment folder not found. Check the experiment name in your YAML config.")

# Save debug results to Drive

In [ ]:
DEBUG_SAVE_DIR = DRIVE_CHECKPOINT_ROOT / "lowlight_nafnet_debug"

!mkdir -p "$DRIVE_CHECKPOINT_ROOT"

if DEBUG_EXP_DIR.exists():
    print("Copying debug experiment to Drive...")
    !rm -rf "$DEBUG_SAVE_DIR"
    !cp -r "$DEBUG_EXP_DIR" "$DEBUG_SAVE_DIR"
    print("Saved to:", DEBUG_SAVE_DIR)
else:
    print("No debug experiment found to copy.")